# Importing libraries

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

from datasets import load_dataset

from sklearn.metrics import precision_score, recall_score, f1_score, hamming_loss
from sklearn.preprocessing import MultiLabelBinarizer
from memory_profiler import memory_usage
from time import perf_counter

# Importing dataset

In [2]:
ds = load_dataset("Rami/multi-label-class-github-issues-text-classification")

train_df = ds['train'].to_pandas()
val_df = ds['valid'].to_pandas()
test_df = ds['test'].to_pandas()

train_df

,title,labels,bodyText
0,TPUs: crash using torch-xla nightly,"[bug, help wanted, won't fix, accelerator: tpu]",🐛 Bug\nIf I try to use torch-xla nightly with ...
1,Fix docs typo in starter files,[docs],"📚 Documentation\nFor typos and doc fixes, plea..."
2,Fix typo in starter files,[docs],"📚 Documentation\nFor typos and doc fixes, plea..."
3,on_*_batch_transfer hooks should include a dat...,"[feature, help wanted]",🚀 Feature\nSee title\nMotivation\nUsers might ...
4,Allow arbitrary val check intervals when using...,"[feature, help wanted]",🚀 Feature\nCurrently when using the max epochs...
...,...,...,...
1551,Logger emits exception when there's `None` in ...,"[bug, help wanted]","To Reproduce\nMy hparams:\n{\n\t'n': [8000],\n..."
1552,TypeError: __init__() got an unexpected keywor...,"[bug, help wanted]","🐛 Bug\nI followed the guide to ""use 16bit prec..."
1553,Simplification: Merge load_from_metrics and lo...,"[feature, help wanted, good first issue]",🚀 Feature\nThe two ways of loading a Lightning...
1554,GPT2-large on Colab TPU seems to time out,"[bug, help wanted]",🐛 Bug\nWhen training gpt2-large on a colab tpu...


# Dataset preprocessing

In [ ]:
allowed_categories = ["bug", "feature", "question", "won't fix", "docs"]

train_df = train_df[train_df['labels'].apply(lambda cats: all(c in allowed_categories for c in cats))]
test_df = test_df[test_df['labels'].apply(lambda cats: all(c in allowed_categories for c in cats))]
val_df = val_df[val_df['labels'].apply(lambda cats: all(c in allowed_categories for c in cats))]

train_df = train_df[train_df['labels'].apply(len) > 0]
test_df = test_df[test_df['labels'].apply(len) > 0]
val_df = val_df[val_df['labels'].apply(len) > 0]

In [4]:
train_df.rename(columns={'title': 'text'}, inplace=True)
test_df.rename(columns={'title': 'text'}, inplace=True)
val_df.rename(columns={'title': 'text'}, inplace=True)

train_df.drop(columns=['bodyText'], inplace=True)
test_df.drop(columns=['bodyText'], inplace=True)
val_df.drop(columns=['bodyText'], inplace=True)

train_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)
val_df.reset_index(drop=True, inplace=True)

train_df

,text,labels
0,Fix docs typo in starter files,[docs]
1,Fix typo in starter files,[docs]
2,Load models give different results from original,[question]
3,Pickle error and OOM when upgrading to 1.2.0,"[question, won't fix]"
4,val_check_interval equivalent for training los...,[won't fix]
...,...,...
410,How to implement pre-training?,[question]
411,Logging the current learning rate,[question]
412,Example of gradient accumulation documentation...,[docs]
413,Checkpooint Callback not called when training ...,[question]


In [6]:
mlb = MultiLabelBinarizer()

train_labels_binarized = mlb.fit_transform(train_df['labels'])
val_labels_binarized = mlb.transform(val_df['labels'])
test_labels_binarized = mlb.transform(test_df['labels'])

train_labels_df = pd.DataFrame(train_labels_binarized, columns=mlb.classes_)
val_labels_df = pd.DataFrame(val_labels_binarized, columns=mlb.classes_)
test_labels_df = pd.DataFrame(test_labels_binarized, columns=mlb.classes_)

train_df = pd.concat([train_df, train_labels_df], axis=1)
val_df = pd.concat([val_df, val_labels_df], axis=1)
test_df = pd.concat([test_df, test_labels_df], axis=1)

train_df = train_df.drop(columns=['labels'])
val_df = val_df.drop(columns=['labels'])
test_df = test_df.drop(columns=['labels'])

train_df

,text,bug,docs,feature,question,won't fix
0,Fix docs typo in starter files,0,1,0,0,0
1,Fix typo in starter files,0,1,0,0,0
2,Load models give different results from original,0,0,0,1,0
3,Pickle error and OOM when upgrading to 1.2.0,0,0,0,1,1
4,val_check_interval equivalent for training los...,0,0,0,0,1
...,...,...,...,...,...,...
410,How to implement pre-training?,0,0,0,1,0
411,Logging the current learning rate,0,0,0,1,0
412,Example of gradient accumulation documentation...,0,1,0,0,0
413,Checkpooint Callback not called when training ...,0,0,0,1,0


In [7]:
class TextDataset(Dataset):
    def __init__(self, texts, label_matrix, tokenizer, max_len):
        """
        texts: a pandas Series or list of strings
        label_matrix: a pandas DataFrame or 2D NumPy array of shape [num_samples, num_labels]
                      Each row i has the 0/1 labels for text i.
        tokenizer: a transformers tokenizer
        max_len: maximum sequence length
        """
        self.texts = texts.tolist()
        # Convert the label matrix into a NumPy array if it isn't already
        self.labels = label_matrix.values if hasattr(label_matrix, 'values') else label_matrix
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        labels = self.labels[idx]  # shape: [num_labels]

        # Tokenize
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=False,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            # Convert labels to float so it works with BCEWithLogitsLoss
            'labels': torch.tensor(labels, dtype=torch.float)
        }


In [8]:
MAX_LEN = 128
BATCH_SIZE = 32

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

label_cols = [col for col in train_df.columns if col != 'text']

# Create datasets
train_dataset = TextDataset(
    texts=train_df['text'],
    label_matrix=train_df[label_cols],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

val_dataset = TextDataset(
    texts=val_df['text'],
    label_matrix=val_df[label_cols],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

test_dataset = TextDataset(
    texts=test_df['text'],
    label_matrix=test_df[label_cols],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# Neural network class (LSTM)

In [9]:
import torch.nn as nn

class LSTMClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout):
        super(LSTMClassifier, self).__init__()

        # For multi-label, output_dim = number_of_labels
        self.embedding = nn.Embedding(tokenizer.vocab_size, embedding_dim)

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            bidirectional=bidirectional,
            batch_first=True,
            dropout=dropout
        )

        # If bidirectional=True, final hidden state has 2*hidden_dim
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids):
        embedded = self.embedding(input_ids)
        outputs, (hidden, cell) = self.lstm(embedded)

        if self.lstm.bidirectional:
            hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        else:
            hidden = hidden[-1,:,:]

        hidden = self.dropout(hidden)
        logits = self.fc(hidden)  # shape [batch_size, output_dim]

        return logits  # raw logits for each label


# Instancing the LSTM model, criterion and optimizer

In [10]:
embedding_dim = 128
hidden_dim = 128
output_dim = len(label_cols)
n_layers = 2
bidirectional = True
dropout = 0.3

model = LSTMClassifier(
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    output_dim=output_dim,
    n_layers=n_layers,
    bidirectional=bidirectional,
    dropout=dropout
)

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using {device} device')
model = model.to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

Using cuda device


# Training and evaluation functions

In [12]:
def train_epoch(model, data_loader, optimizer, criterion, device):
    model.train()
    losses = []
    correct_predictions = 0
    total_labels = 0

    all_labels = []
    all_preds = []

    for batch in data_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)  # shape: (batch_size, num_labels)

        optimizer.zero_grad()

        # Forward pass -> logits: [batch_size, num_labels]
        logits = model(input_ids)

        # Compute loss
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        # Convert logits to predictions in {0,1}
        preds = (torch.sigmoid(logits) > 0.5).float()

        # Count how many individual labels are predicted correctly
        correct_predictions += (preds == labels).sum().item()
        total_labels += labels.numel()

        # Store for metric calculation
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

    # Calculate mean loss
    avg_loss = sum(losses) / len(losses)
    # Label-level accuracy
    accuracy = correct_predictions / total_labels

    # Convert to NumPy
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)

    # Macro-average precision, recall, F1
    precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    # Per-class F1
    f1_per_class = f1_score(all_labels, all_preds, average=None, zero_division=0)

    # Hamming loss
    ham_loss = hamming_loss(all_labels, all_preds)

    return accuracy, avg_loss, precision, recall, f1_macro, f1_per_class, ham_loss


def eval_model(model, data_loader, criterion, device):
    model.eval()
    losses = []
    correct_predictions = 0
    total_labels = 0

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids)
            loss = criterion(logits, labels)
            losses.append(loss.item())

            preds = (torch.sigmoid(logits) > 0.5).float()

            correct_predictions += (preds == labels).sum().item()
            total_labels += labels.numel()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    avg_loss = sum(losses) / len(losses)
    accuracy = correct_predictions / total_labels

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)

    precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    f1_per_class = f1_score(all_labels, all_preds, average=None, zero_division=0)
    ham_loss = hamming_loss(all_labels, all_preds)

    return accuracy, avg_loss, precision, recall, f1_macro, f1_per_class, ham_loss


# Training loop

In [13]:
def training_loop(epochs):
    for epoch in range(epochs):
        print(f'Epoch {epoch + 1}/{epochs}')
        
        (
            train_acc, 
            train_loss, 
            train_prec, 
            train_rec, 
            train_f1_macro, 
            train_f1_per_class,
            train_ham_loss
        ) = train_epoch(model, train_loader, optimizer, criterion, device)
        
        (
            val_acc, 
            val_loss, 
            val_prec, 
            val_rec, 
            val_f1_macro, 
            val_f1_per_class,
            val_ham_loss
        ) = eval_model(model, val_loader, criterion, device)
        
        print(f"Train Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}, "
              f"Precision(macro): {train_prec:.4f}, Recall(macro): {train_rec:.4f}, "
              f"F1(macro): {train_f1_macro:.4f}, Hamming: {train_ham_loss:.4f}")
        print(f"F1 Per Class (Train): {train_f1_per_class}")
        
        print(f"Val   Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}, "
              f"Precision(macro): {val_prec:.4f}, Recall(macro): {val_rec:.4f}, "
              f"F1(macro): {val_f1_macro:.4f}, Hamming: {val_ham_loss:.4f}")
        print(f"F1 Per Class (Val):   {val_f1_per_class}")
        print("--------------------------------------------------")
    
    return (
        train_acc, train_loss, train_prec, train_rec, train_f1_macro, train_f1_per_class, train_ham_loss,
        val_acc,   val_loss,   val_prec,   val_rec,   val_f1_macro,   val_f1_per_class,   val_ham_loss
    )


In [14]:

seeds = [2, 3, 5]
EPOCHS = 5

# Update your results DataFrame with Hamming Loss columns
results = pd.DataFrame(columns=[
    'seed',
    'train_loss', 'train_acc', 'train_prec', 'train_rec', 'train_f1', 'train_f1_per_class', 'train_ham',
    'val_loss',   'val_acc',   'val_prec',   'val_rec',   'val_f1',   'val_f1_per_class',   'val_ham',
    'test_loss',  'test_acc',  'test_prec',  'test_rec',  'test_f1',  'test_f1_per_class',  'test_ham',
    'max_memory_usage_train', 'max_vram_usage_train', 'total_time_train',
    'max_memory_usage_test',  'max_vram_usage_test',  'total_time_test'
])

for seed in seeds:
    torch.manual_seed(seed)
    
    # Reset / re-initialize model for each seed
    model = LSTMClassifier(
        embedding_dim=embedding_dim,
        hidden_dim=hidden_dim,
        output_dim=len(label_cols),  # Number of labels for multi-label
        n_layers=n_layers,
        bidirectional=bidirectional,
        dropout=dropout
    ).to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    # For multi-label classification, use BCEWithLogitsLoss
    criterion = nn.BCEWithLogitsLoss().to(device)
    
    # Reset CUDA memory tracking if using GPU
    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    # -------- TRAINING -----------
    start_time_train = perf_counter()
    
    max_memory_usage_train, retval = memory_usage(
        (training_loop, (EPOCHS,), {}),
        retval=True, 
        max_usage=True
    )
    total_time_train = perf_counter() - start_time_train

    max_vram_usage_train = (
        torch.cuda.max_memory_allocated() / (1024 ** 2)
        if torch.cuda.is_available() else None
    )

    (
        train_acc, train_loss, train_prec, train_rec, train_f1_macro, train_f1_per_class, train_ham,
        val_acc,   val_loss,   val_prec,   val_rec,   val_f1_macro,   val_f1_per_class,   val_ham
    ) = retval

    # Reset CUDA memory tracking before test evaluation
    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    # -------- TESTING -----------
    start_time_test = perf_counter()
    # eval_model should return:
    # (test_acc, test_loss, test_prec, test_rec, test_f1, test_f1_per_class, test_ham)
    max_memory_usage_test, retval = memory_usage(
        (eval_model, (model, test_loader, criterion, device), {}),
        retval=True, 
        max_usage=True
    )
    total_time_test = perf_counter() - start_time_test

    max_vram_usage_test = (
        torch.cuda.max_memory_allocated() / (1024 ** 2)
        if torch.cuda.is_available() else None
    )

    test_acc, test_loss, test_prec, test_rec, test_f1, test_f1_per_class, test_ham = retval

    # -------- LOGGING -----------
    new_row = pd.DataFrame([[
        seed,
        train_loss, train_acc, train_prec, train_rec, train_f1_macro, train_f1_per_class, train_ham,
        val_loss,   val_acc,   val_prec,   val_rec,   val_f1_macro,   val_f1_per_class,   val_ham,
        test_loss,  test_acc,  test_prec,  test_rec,  test_f1,        test_f1_per_class,  test_ham,
        max_memory_usage_train, max_vram_usage_train, total_time_train,
        max_memory_usage_test,  max_vram_usage_test,  total_time_test
    ]], columns=results.columns)

    results = pd.concat([results, new_row], ignore_index=True)

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 0.5268, Accuracy: 0.7696, Precision(macro): 0.2118, Recall(macro): 0.2496, F1(macro): 0.2146, Hamming: 0.2304
F1 Per Class (Train): [0.04958678 0.         0.14634146 0.74923547 0.128     ]
Val   Loss: 0.4576, Accuracy: 0.8128, Precision(macro): 0.1294, Recall(macro): 0.2000, F1(macro): 0.1571, Hamming: 0.1872
F1 Per Class (Val):   [0.         0.         0.         0.78571429 0.        ]
--------------------------------------------------
Epoch 2/5
Train Loss: 0.4506, Accuracy: 0.8043, Precision(macro): 0.1235, Recall(macro): 0.1992, F1(macro): 0.1525, Hamming: 0.1957
F1 Per Class (Train): [0.         0.         0.         0.76233184 0.        ]
Val   Loss: 0.4489, Accuracy: 0.8128, Precision(macro): 0.1294, Recall(macro): 0.2000, F1(macro): 0.1571, Hamming: 0.1872
F1 Per Class (Val):   [0.         0.         0.         0.78571429 0.        ]
--------------------------------------------------
Epoch 3/5
Train Loss: 0.4260, Accuracy: 0.8087, Precision(macro): 0.1288, 

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
C:\Users\Rafael\AppData\Local\Temp\ipykernel_20284\3822712177.py:87: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, new_row], ignore_index=True)
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 0.5233, Accuracy: 0.7667, Precision(macro): 0.2388, Recall(macro): 0.2549, F1(macro): 0.2300, Hamming: 0.2333
F1 Per Class (Train): [0.08264463 0.         0.09302326 0.73716952 0.23703704]
Val   Loss: 0.4606, Accuracy: 0.8128, Precision(macro): 0.1294, Recall(macro): 0.2000, F1(macro): 0.1571, Hamming: 0.1872
F1 Per Class (Val):   [0.         0.         0.         0.78571429 0.        ]
--------------------------------------------------
Epoch 2/5
Train Loss: 0.4494, Accuracy: 0.7995, Precision(macro): 0.1263, Recall(macro): 0.1633, F1(macro): 0.1424, Hamming: 0.2005
F1 Per Class (Train): [0.        0.        0.        0.7120954 0.       ]
Val   Loss: 0.4408, Accuracy: 0.8128, Precision(macro): 0.1294, Recall(macro): 0.2000, F1(macro): 0.1571, Hamming: 0.1872
F1 Per Class (Val):   [0.         0.         0.         0.78571429 0.        ]
--------------------------------------------------
Epoch 3/5
Train Loss: 0.4216, Accuracy: 0.8082, Precision(macro): 0.1277, Recal

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 0.5196, Accuracy: 0.7841, Precision(macro): 0.1867, Recall(macro): 0.2011, F1(macro): 0.1732, Hamming: 0.2159
F1 Per Class (Train): [0.0754717  0.04819277 0.         0.74213836 0.        ]
Val   Loss: 0.4565, Accuracy: 0.8128, Precision(macro): 0.1294, Recall(macro): 0.2000, F1(macro): 0.1571, Hamming: 0.1872
F1 Per Class (Val):   [0.         0.         0.         0.78571429 0.        ]
--------------------------------------------------
Epoch 2/5
Train Loss: 0.4426, Accuracy: 0.8077, Precision(macro): 0.1256, Recall(macro): 0.1992, F1(macro): 0.1541, Hamming: 0.1923
F1 Per Class (Train): [0.         0.         0.         0.77039275 0.        ]
Val   Loss: 0.4414, Accuracy: 0.8107, Precision(macro): 0.1286, Recall(macro): 0.1967, F1(macro): 0.1556, Hamming: 0.1893
F1 Per Class (Val):   [0.         0.         0.         0.77777778 0.        ]
--------------------------------------------------
Epoch 3/5
Train Loss: 0.4142, Accuracy: 0.8145, Precision(macro): 0.1321, 

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


In [15]:
results.to_csv('results/lstm_multilabel2.csv', index=False)
results.head()

,seed,train_loss,train_acc,train_prec,train_rec,train_f1,train_f1_per_class,train_ham,val_loss,val_acc,...,test_rec,test_f1,test_f1_per_class,test_ham,max_memory_usage_train,max_vram_usage_train,total_time_train,max_memory_usage_test,max_vram_usage_test,total_time_test
0,2,0.333872,0.857831,0.472605,0.292401,0.324200,"[0.5569620253164557, 0.20689655172413793, 0.0,...",0.142169,0.438974,0.818182,...,0.265530,0.303948,"[0.30303030303030304, 0.41025641025641024, 0.0...",0.168,1209.332031,231.645020,1.718039,1214.574219,194.687988,1.035454
1,3,0.342548,0.849157,0.502301,0.179369,0.205808,"[0.06315789473684211, 0.1111111111111111, 0.0,...",0.150843,0.430243,0.837433,...,0.177771,0.174815,"[0.0, 0.07407407407407407, 0.0, 0.8, 0.0]",0.174,1214.757812,232.631348,1.497684,1214.781250,195.408691,1.140147
2,5,0.330751,0.857831,0.525494,0.231869,0.267868,"[0.3559322033898305, 0.1111111111111111, 0.0, ...",0.142169,0.415574,0.825668,...,0.218021,0.242208,"[0.25, 0.14285714285714285, 0.0, 0.81818181818...",0.165,1229.816406,232.510254,1.787512,1229.832031,194.881348,1.116701


In [16]:
torch.save(model.state_dict(), 'results/lstm_multilabel2.pth')